In [ ]:
import numpy as np
import pandas as pd   #data wrnglin=g library

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px  # visualization libraries

In [ ]:
df=pd.read_csv('/content/googleplaystore_v2.csv')

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.isna().sum()  #wherever null values are there sum them and show me the nulls i every column

In [ ]:
#install->int
#price ->float
#reviews->int

In [ ]:
df[df['Reviews']=='3.0M']

In [ ]:
df=df.drop(10472)

In [ ]:
df['Reviews']=df['Reviews'].astype('int')

In [ ]:
df.info()

In [ ]:
def missing_values_table(df):

  #total missing values
  mis_val=df.isna().sum()

  #percentage of missing
  mis_val_percent=mis_val/len(df)*100

  #make a table with the results
  mis_val_table=pd.concat([mis_val,mis_val_percent],axis=1)

  #rename the table columns
  mis_val_table.columns=['Missing Values','Missing Values %']

  #sort the table by mising values
  sorted_missing_val_table=mis_val_table.sort_values('Missing Values',ascending=False).round(1)

  #print summary information
  print('Your selected dataframe has '+str(df.shape[1])+' columns.\n'
        'There are '+str(df.shape[0])+' and these many rows')

  return sorted_missing_val_table

In [ ]:
missing_values_table(df)

In [ ]:
df=df[~df['Rating'].isna()]

In [ ]:
missing_values_table(df)

In [ ]:
df[df['Current Ver'].isna()]

In [ ]:
df[df['Current Ver'].isna()]

In [ ]:
df[df['Android Ver'].isna()]

In [ ]:
df[df['Category']=='PERSONALIZATION']['Android Ver'].mode()

In [ ]:
personalization_apps=df[df['Category']=='PERSONALIZATION']

In [ ]:
personalization_apps['Android Ver'].mode()

In [ ]:
df['Android Ver']=df['Android Ver'].fillna('4.0.3 and up')

In [ ]:
df[df['Android Ver'].isna()]

In [ ]:
df[df['Category']=="ART_AND_DESIGN"]['Current Ver'].mode()

In [ ]:
df[df['Category']=="LIBRARIES_AND_DEMO"]['Current Ver'].mode()

In [ ]:
df[df['Category']=="TOOLS"]['Current Ver'].mode()

In [ ]:
df[df['Category']=="FAMILY"]['Current Ver'].mode()

In [ ]:
df.loc[[15,1553,6322,7333],"Current Ver"]=['1.0','2.3.2','Varies with device','1.0']   # loc function has row_indexes,columnindex  either number of rows are more witha single column OR below case

In [ ]:
#df.loc[15,["Current Ver","Android Ver"]=['1.0','2.3.2']
# loc function has row_indexes,columnindex  either number of row is one where multiple c0olumns can be filled or both

In [ ]:
missing_values_table(df)

In [ ]:
def clean_installs(val):
  return int(val.replace(',','').replace('+',''))

In [ ]:
df['Installs']=df['Installs'].apply(clean_installs)

In [ ]:
df.info()

In [ ]:
df['Price']=df['Price'].apply(lambda x:0 if x=='0'else float(x[1:]))

In [ ]:
df.info()

In [ ]:
df['Price'].value_counts()

In [ ]:
#Sanity check1
# if an app is of type=free then the price column should not have any value apart from 0

#sanity check 2
#ALL APS SHOUKD HAVE RATING BETWEEN  1 AND 5

#SANITY CHECK 3
# NO FO REVIEW SHOUDL NEVER BE MORE THAN NUMBER OF INSTALLS

In [ ]:
#Sanity check1
# if an app is of type=free then the price column should not have any value apart from 0
df[(df['Type']=='Free')&(df['Price']>0)]

In [ ]:
#sanity check 2
#ALL APS SHOUKD HAVE RATING BETWEEN  1 AND 5
df[~((df['Rating']>=1) & (df['Rating']<=5))]

In [ ]:
#SANITY CHECK 3
# NO FO REVIEW SHOUDL NEVER BE MORE THAN NUMBER OF INSTALLS
df=df[(df['Reviews'])<=(df['Installs'])]

In [ ]:
df.shape

In [ ]:
df['Type'].value_counts()

In [ ]:
#does paid apps do better than free apps ,and does outliers in the paid apps have an advantage
#boxplot for outlier analysis on price
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=df, x="Type", y="Rating")
# Tweak: change bins, add stat="density", or remove kde=True

In [ ]:
fig = px.violin(data_frame=df, x="Type", y="Rating", title="Rating Distribution by App Type")
fig.show()

In [ ]:
#hypothesis testing using t-Test 2 sample
#t-test checks wether means of two groups are statistically different from each other also calle as welch's Test
#1.ho(null hypotheis) mean rating of free apps=mean rating rating of paid apps
#H1(alternate hypothesis) mean rating of free apps!=mean rating rating of paid apps
#p=0.05
#decision reject Ho if p-value<alpha

#p>0.05 hypothesis is weak
#p<0.05 hypothesis is significant
#p<0.01 hypothesis is strong
#p<0.001 hypothesis is very very strong

In [ ]:
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols

In [ ]:
free_Rating=df[df['Type']=='Free']['Rating']
paid_Rating=df[df['Type']=='Paid']['Rating']

In [ ]:
print(free_Rating.mean())
print(free_Rating.std())
len(free_Rating)

In [ ]:
print(paid_Rating.mean())
print(paid_Rating.std())
len(paid_Rating)

In [ ]:
t_stat,p_stat=stats.ttest_ind(free_Rating,paid_Rating,equal_var=False)

In [ ]:
print(t_stat)
print(p_stat)

In [ ]:
# Get top 10 categories for Free apps based on count
top_10_free_categories = df[df['Type'] == 'Free']['Category'].value_counts().nlargest(10).index.tolist()

# Get top 10 categories for Paid apps based on count
top_10_paid_categories = df[df['Type'] == 'Paid']['Category'].value_counts().nlargest(10).index.tolist()

# Filter DataFrame for Free apps with their top 10 categories
df_free_filtered = df[(df['Type'] == 'Free') & (df['Category'].isin(top_10_free_categories))]

# Filter DataFrame for Paid apps with their top 10 categories
df_paid_filtered = df[(df['Type'] == 'Paid') & (df['Category'].isin(top_10_paid_categories))]

# Concatenate the filtered dataframes
df_filtered = pd.concat([df_free_filtered, df_paid_filtered])

plt.figure(figsize=(16, 8))
sns.violinplot(data=df_filtered, x="Type", y="Rating", hue="Category", inner="quartile", palette="tab20")
plt.title("Rating Distribution by App Type and Top 10 Categories")
plt.xlabel("App Type")
plt.ylabel("Rating")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
mean_ratings = df.groupby(['Category', 'Type'])['Rating'].mean().reset_index()
plt.figure(figsize=(16, 8))
sns.barplot(data=mean_ratings, x='Category', y='Rating', hue='Type', palette='viridis')
plt.title('Average Rating by App Category and Type')
plt.xlabel('Category')
plt.ylabel('Average Rating')
plt.xticks(rotation=90, ha='right')
plt.legend(title='App Type')
plt.tight_layout()
plt.show()

In [ ]:
mean_ratings = df.groupby(['Category', 'Type'])['Rating'].mean()
mean_ratings

In [ ]:
#t-test 2 sample
t_stat,p_stat=stats.ttest_ind(free_Rating,paid_Rating,equal_var=False)